In [ ]:
# ============================================
# Phase 2 — Day 11: Edge Detection
# Gradients, Sobel, Laplacian, Canny
# ============================================

import cv2 
import numpy as np
import matplotlib.pyplot as plt 

# Load our test image 
img = cv2.imread('/home/arpeetpadhy/test_image.png')
gray =  cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

print("Image loaded !")
print("Shape : ", gray.shape)
print("Ready for edge detection ! 🚀")

In [ ]:
# ============================================
# SOBEL EDGE DETECTION — Gradient Based
# ============================================

# Sobel in X direction -- detects vertical edges 
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize =3 )

# Sobel inY direction -- detects horizantal edges 
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize = 3)

#combined magnitude - detects all edges 
sobel_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)
sobel_mag = np.uint8(np.clip(sobel_mag, 0, 255))

fig, axes = plt.subplots(1,4, figsize = (16, 4))

axes[0].imshow(gray, cmap = 'gray')
axes[0].set_title(' Original Grayscale')
axes[0].axis('off')

axes[1].imshow(np.abs(sobel_x), cmap='gray')
axes[1].set_title('Sobel X\n(vertical edges)')
axes[1].axis('off')

axes[2].imshow(np.abs(sobel_y), cmap='gray')
axes[2].set_title('Sobel Y\n(horizontal edges)')
axes[2].axis('off')

axes[3].imshow(sobel_mag, cmap='gray')
axes[3].set_title('Sobel Combined\n(all edges)')
axes[3].axis('off')

plt.suptitle('Sobel Edge Detection — Gradient Based', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Sobel X detects vertical edges (changes in X direction)")
print("Sobel Y detects horizontal edges (changes in Y direction)")
print("Combined magnitude = sqrt(Sx² + Sy²) detects all edges")

In [ ]:
# ============================================
# LAPLACIAN EDGE DETECTION — Second-Order Derivative
# ============================================
# Unlike Sobel (first derivative, direction-specific), Laplacian is
# a single second-derivative operator — it doesn't care about direction,
# it just finds where intensity change is changing fastest (zero-crossings)


laplacian = cv2.Laplacian(gray, cv2.CV_64F, ksize = 3)
laplacian_abs = np.uint8(np.clip(np.abs(laplacian),0 ,255))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(gray, cmap ='gray')
axes[0].set_title('Original Grayscale')
axes[0].axis('off')

axes[1].imshow(laplacian, cmap='gray')
axes[1].set_title('Laplacian (raw)\n(signed, +/- crossings)')
axes[1].axis('off')

axes[2].imshow(laplacian_abs, cmap='gray')
axes[2].set_title('Laplacian (abs)\n(edge strength)')
axes[2].axis('off')

plt.suptitle('Laplacian Edge Detection — Second-Order', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Laplacian = ∂²I/∂x² + ∂²I/∂y² — a single omnidirectional operator")
print("Downside: very sensitive to noise (it's a 2nd derivative) — usually blurred first")
print("That noise sensitivity is exactly why Canny exists — coming up next 👇")

In [ ]:
# ============================================
# CANNY EDGE DETECTION — Multi-Stage Pipeline
# ============================================
# Canny = Gaussian blur (noise reduction) -> Sobel gradients -> 
#         Non-max suppression (thin edges to 1px) -> Hysteresis thresholding

# Step 1: blur first — Canny is noise-sensitive just like Laplacian
blurred = cv2.GaussianBlur(gray, (5,5), sigmaX = 1.4)

# Step 2-4 handled internally by cv2.Canny — we supply two thresholds:
# below low_threshold  -> definitely not an edge
# above high_threshold -> definitely an edge
# in between           -> edge only if connected to a "definite" edge (hysteresis)

low_threshold = 50 
high_threshold = 150 
edges_canny = cv2.Canny(blurred, low_threshold, high_threshold)

fig, axes = plt.subplots(1, 4, figsize = (16 ,4))

axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Original Grayscale')
axes[0].axis('off')

axes[1].imshow(sobel_mag, cmap='gray')
axes[1].set_title('Sobel\n(thick, noisy edges)')
axes[1].axis('off')

axes[2].imshow(laplacian_abs, cmap='gray')
axes[2].set_title('Laplacian\n(very noisy)')
axes[2].axis('off')

axes[3].imshow(edges_canny, cmap='gray')
axes[3].set_title('Canny\n(thin, clean edges)')
axes[3].axis('off')

plt.suptitle('Edge Detection Comparison: Sobel vs Laplacian vs Canny', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Canny pipeline: Gaussian blur -> Sobel gradient -> non-max suppression -> hysteresis")
print(f"Thresholds used: low={low_threshold}, high={high_threshold}")
print("Try changing these two numbers — that's the fastest way we build intuition for Canny")

In [ ]:
# ============================================
# THRESHOLD SENSITIVITY — Canny in action
# ============================================
thresholds = [(30, 90), (50, 150), (100, 200)]

fig, axes = plt.subplots(1, len(thresholds), figsize=(4*len(thresholds), 4))
for ax, (low, high) in zip(axes, thresholds):
    e = cv2.Canny(blurred, low, high)
    ax.imshow(e, cmap='gray')
    ax.set_title(f'low={low}, high={high}')
    ax.axis('off')

plt.suptitle('Canny is a two-knob system — feel out the knobs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# quick check: what's the actual gradient magnitude at that boundary?
print("Max Sobel magnitude overall:", sobel_mag.max())
print("Sobel magnitude in bottom-right region:", sobel_mag[gray.shape[0]//2:, gray.shape[1]//2:].max())